<a href="https://colab.research.google.com/github/annatsamoyra-prog/data-story-/blob/main/dnews_scrapper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time
import random
import logging

from datetime import datetime
from urllib.parse import urljoin
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [2]:
BASE_URL = "https://www.dnews.gr"

CATEGORY_PAGES = [
    "https://www.dnews.gr/eidhseis/texnologia?start=124",
    "https://www.dnews.gr/eidhseis/texnologia?start=93",
    "https://www.dnews.gr/eidhseis/texnologia?start=62",
    "https://www.dnews.gr/eidhseis/texnologia?start=31",
    "https://www.dnews.gr/eidhseis/texnologia?start=1"
]

# Το άρθρο που θέλουμε οπωσδήποτε να συμπεριληφθεί
EXTRA_ARTICLES = [
    "https://www.dnews.gr/eidhseis/texnologia/566340/i-texniti-noimosyni-metamorfonei-ton-tropo-pou-katanalonoume-eidiseis"
]

SOURCE_NAME = "Dnews"

MIN_DELAY = 1.0
MAX_DELAY = 2.0
REQUEST_TIMEOUT = 20

OUTPUT_CSV = "dnews_texnologia.csv"

In [3]:

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    ),
    "Accept-Language": "el-GR,el;q=0.9,en-US;q=0.8,en;q=0.7",
}


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)

log = logging.getLogger("dnews_scraper")


def build_session():

    session = requests.Session()
    session.headers.update(HEADERS)

    retries = Retry(
        total=3,
        backoff_factor=1.0,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )

    session.mount(
        "https://",
        HTTPAdapter(max_retries=retries)
    )

    session.mount(
        "http://",
        HTTPAdapter(max_retries=retries)
    )

    return session


In [4]:
def is_dnews_technology_article(url):

    if not url:
        return False

    # Κρατάμε μόνο URLs άρθρων της κατηγορίας τεχνολογία.
    # Τα άρθρα έχουν αριθμητικό ID μετά το /texnologia/
    return bool(
        re.search(
            r"dnews\.gr/eidhseis/texnologia/\d+/",
            url
        )
    )


In [5]:
def collect_article_urls():

    session = build_session()

    article_urls = set()

    for page_url in CATEGORY_PAGES:

        log.info("Διαβάζω category page: %s", page_url)

        try:

            response = session.get(
                page_url,
                timeout=REQUEST_TIMEOUT
            )

            response.raise_for_status()

        except requests.RequestException as exc:

            log.warning(
                "Αποτυχία στη σελίδα %s: %s",
                page_url,
                exc
            )

            continue


        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )


        # Ψάχνουμε όλους τους συνδέσμους.
        # Μετά κρατάμε μόνο όσους είναι άρθρα τεχνολογίας.

        for a in soup.find_all("a", href=True):

            href = a.get("href")

            full_url = urljoin(
                BASE_URL,
                href
            )

            # αφαιρούμε fragments
            full_url = full_url.split("#")[0]

            if is_dnews_technology_article(full_url):

                article_urls.add(full_url)


        log.info(
            "Μέχρι τώρα βρέθηκαν %d μοναδικά άρθρα",
            len(article_urls)
        )

        time.sleep(
            random.uniform(
                MIN_DELAY,
                MAX_DELAY
            )
        )


    # Προσθέτουμε και το συγκεκριμένο άρθρο
    # που μας έδωσες.

    for url in EXTRA_ARTICLES:
        article_urls.add(url)


    article_urls = sorted(article_urls)

    log.info(
        "Συνολικά URLs άρθρων: %d",
        len(article_urls)
    )

    return article_urls

In [6]:
DATE_PATTERN = re.compile(
    r"(\d{1,2})\.(\d{1,2})\.(\d{4})"
    r"(?:\s+(\d{1,2}):(\d{2}))?"
)


def parse_date(text):

    if not text:
        return None

    match = DATE_PATTERN.search(text)

    if not match:
        return None

    day, month, year, hour, minute = match.groups()

    try:

        if hour and minute:

            return datetime(
                int(year),
                int(month),
                int(day),
                int(hour),
                int(minute)
            )

        return datetime(
            int(year),
            int(month),
            int(day)
        )

    except ValueError:

        return None

In [7]:

def extract_article(session, url):

    try:

        response = session.get(
            url,
            timeout=REQUEST_TIMEOUT
        )

        response.raise_for_status()

    except requests.RequestException as exc:

        log.warning(
            "Αποτυχία άρθρου %s: %s",
            url,
            exc
        )

        return None


    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )


    # --------------------------------------------------------
    # TITLE
    # --------------------------------------------------------

    h1 = soup.find("h1")

    if h1:

        title = h1.get_text(
            " ",
            strip=True
        )

    else:

        # fallback στο OpenGraph title

        og_title = soup.find(
            "meta",
            property="og:title"
        )

        title = (
            og_title.get("content", "").strip()
            if og_title
            else None
        )


    # --------------------------------------------------------
    # DATE
    # --------------------------------------------------------

    date_value = None

    # Πρώτα δοκιμάζουμε structured metadata

    time_tag = soup.find("time")

    if time_tag:

        date_value = parse_date(
            time_tag.get_text(
                " ",
                strip=True
            )
        )


    # fallback: ψάχνουμε στο κείμενο της σελίδας

    if date_value is None:

        page_text = soup.get_text(
            " ",
            strip=True
        )

        date_value = parse_date(page_text)


    # --------------------------------------------------------
    # AUTHOR
    # --------------------------------------------------------

    author = None

    # meta author
    meta_author = soup.find(
        "meta",
        attrs={"name": "author"}
    )

    if meta_author:

        author = meta_author.get(
            "content",
            ""
        ).strip()


    # εναλλακτικοί selectors

    if not author:

        author_element = soup.find(
            attrs={
                "class": re.compile(
                    r"(author|itemAuthor|createdby)",
                    re.I
                )
            }
        )

        if author_element:

            author = author_element.get_text(
                " ",
                strip=True
            )


    # --------------------------------------------------------
    # FULL TEXT
    # --------------------------------------------------------

    full_text = extract_full_text(soup)


    return {

        "site": SOURCE_NAME,

        "url": url,

        "title": title,

        "date": (
            date_value.date()
            if date_value
            else None
        ),

        "datetime": date_value,

        "author": author,

        "full_text": full_text
    }

In [8]:
STOP_PHRASES = [

    "ΟΙ ΕΙΔΗΣΕΙΣ ΣΕ 2",

    "Δες όλες τις ειδήσεις",

    "# TAGS",

    "Copyright",

    "ΡΟΗ ΕΙΔΗΣΕΩΝ"
]


def extract_full_text(soup):

    # --------------------------------------------------------
    # Προσπαθούμε πρώτα να εντοπίσουμε το κύριο άρθρο
    # --------------------------------------------------------

    article = soup.find("article")


    # Εναλλακτικά containers

    if article is None:

        article = soup.find(
            "div",
            class_=re.compile(
                r"(itemFullText|item-fulltext|article-content|"
                r"itemBody|articleBody|post-content)",
                re.I
            )
        )


    # fallback
    search_root = (
        article
        if article is not None
        else soup
    )


    paragraphs = search_root.find_all("p")


    text_parts = []


    for p in paragraphs:

        text = p.get_text(
            " ",
            strip=True
        )


        if not text:
            continue


        # Πολύ μικρά στοιχεία συνήθως είναι UI
        if len(text) < 20:
            continue


        # Σταματάμε σε sections που δεν ανήκουν
        # στο κυρίως άρθρο

        if any(
            phrase.lower() in text.lower()
            for phrase in STOP_PHRASES
        ):

            break


        text_parts.append(text)


    # --------------------------------------------------------
    # Αφαίρεση διπλών paragraphs
    # --------------------------------------------------------

    unique_parts = []

    seen = set()


    for text in text_parts:

        if text not in seen:

            seen.add(text)

            unique_parts.append(text)


    full_text = "\n\n".join(
        unique_parts
    ).strip()


    return (
        full_text
        if full_text
        else None
    )

In [9]:

def scrape():

    urls = collect_article_urls()

    session = build_session()

    articles = []

    total = len(urls)


    for i, url in enumerate(
        urls,
        start=1
    ):

        log.info(
            "Άρθρο %d/%d: %s",
            i,
            total,
            url
        )


        article = extract_article(
            session,
            url
        )


        if article is not None:

            articles.append(article)


        time.sleep(
            random.uniform(
                MIN_DELAY,
                MAX_DELAY
            )
        )


    df = pd.DataFrame(articles)


    if not df.empty:

        df = (
            df
            .drop_duplicates(
                subset=["url"]
            )
            .reset_index(drop=True)
        )


    return df

In [10]:
df_dnews = scrape()

print(
    "Συνολικά άρθρα:",
    len(df_dnews)
)

display(
    df_dnews.head()
)

Συνολικά άρθρα: 149


,site,url,title,date,datetime,author,full_text
0,Dnews,https://www.dnews.gr/eidhseis/texnologia/56634...,Η Τεχνητή Νοημοσύνη μεταμορφώνει τον τρόπο που...,2026-01-12,2026-01-12 10:22:00,Newsroom,Οι εταιρείες μέσων ενημέρωσης αναμένουν ότι η ...
1,Dnews,https://www.dnews.gr/eidhseis/texnologia/56652...,«Είσαι νεκρός;»: Η viral κινεζική εφαρμογή για...,2026-01-13,2026-01-13 08:29:00,Newsroom,Φέρει την ονομασία «Are you dead?» («Είσαι Νεκ...
2,Dnews,https://www.dnews.gr/eidhseis/texnologia/56685...,Κρισταλίνα Γκεοργκίεβα: Το μέλλον της εργασίας...,2026-01-14,2026-01-14 16:19:00,Θανάσης Κουκάκης,Σε μια περίοδο έντονων ανακατατάξεων στην παγκ...
3,Dnews,https://www.dnews.gr/eidhseis/texnologia/56687...,Samsung Galaxy S26: Στις 25 Φεβρουαρίου η παρο...,2026-01-14,2026-01-14 17:33:00,Newsroom,H Samsung φαίνεται ότι κάνει τις τελευταίες πρ...
4,Dnews,https://www.dnews.gr/eidhseis/texnologia/56754...,Νέα γενιά ψηφιακών βοηθών: Τι υπόσχεται το εξα...,2026-01-18,2026-01-18 16:44:00,Newsroom,"Η τεχνητή νοημοσύνη περνά σε μια νέα φάση, όπο..."


In [11]:
df_dnews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 149 entries, 0 to 148
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   site       149 non-null    object        
 1   url        149 non-null    object        
 2   title      149 non-null    object        
 3   date       149 non-null    object        
 4   datetime   149 non-null    datetime64[ns]
 5   author     149 non-null    object        
 6   full_text  149 non-null    object        
dtypes: datetime64[ns](1), object(6)
memory usage: 8.3+ KB


In [12]:
#πόσα άρθρα δεν πήραν κείμενο:
df_dnews['full_text'].isna().sum()

np.int64(0)

In [13]:
df_dnews['date'].min(), df_dnews['date'].max()

(datetime.date(2026, 1, 12), datetime.date(2026, 9, 17))

In [14]:
df_dnews.to_csv(
    "dnews_texnologia.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Αποθηκεύτηκε το dnews_texnologia.csv")

Αποθηκεύτηκε το dnews_texnologia.csv


In [23]:
import base64
from google.colab import userdata

In [24]:
from google.colab import userdata

save_df_to_github(
    df_dnews,
    repo="annatsamoyra-prog/data-story-",
    path="dnews_texnologia_2026.csv",
    token=userdata.get("newtoken"),
)

Successfully saved dnews_texnologia_2026.csv to GitHub repository annatsamoyra-prog/data-story-
